---
title: "Trabalho PLN II"
author: "Douglas e Thiago"
format: html
---

### Contextualização do projeto

O projeto consiste em desenvolver modelos de busca comparativos. O dataset escolhido para exploração foi retirado do Kaggle e é sobre opiniões em produtos de diversas plataformas de e-commerce do Brasil.

### 1 - Importação de bibliotecas e load do dataset

In [2]:
import kagglehub
import os
import pandas as pd
import numpy as np
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

path = kagglehub.dataset_download("fredericods/ptbr-sentiment-analysis-datasets")

print("Path to dataset files:", path)
print(os.listdir(path))

Path to dataset files: C:\Users\Usuario\.cache\kagglehub\datasets\fredericods\ptbr-sentiment-analysis-datasets\versions\1
['b2w.csv', 'buscape.csv', 'concatenated.csv', 'olist.csv', 'utlc_apps.csv', 'utlc_movies.csv']


In [3]:
#Carregamos o arquivo dentro da variável df
arquivo = path + "/b2w.csv"

df = pd.read_csv(arquivo)
df.shape
df.head()

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,11955,Bem macio e felpudo...recomendo. Preço imbatí...,bem macio e felpudo...recomendo. preco imbati...,"['bem', 'macio', 'felpudo', 'recomendo', 'prec...",1.0,4,1,1
1,35478,Produto excepcional! recomendo!!! inovador e ...,produto excepcional! recomendo!!! inovador e ...,"['produto', 'excepcional', 'recomendo', 'inova...",1.0,5,1,1
2,122760,recebi o produto antes do prazo mas veio com d...,recebi o produto antes do prazo mas veio com d...,"['recebi', 'produto', 'antes', 'do', 'prazo', ...",0.0,1,1,1
3,17114,Bom custo beneficio. Adequado para pessoas que...,bom custo beneficio. adequado para pessoas que...,"['bom', 'custo', 'beneficio', 'adequado', 'par...",1.0,5,1,1
4,19112,Além de higiênico tem o tamanho ideal. Só falt...,alem de higienico tem o tamanho ideal. so falt...,"['alem', 'de', 'higienico', 'tem', 'tamanho', ...",NaN,3,-1,1


### 1.2 - Drop de valores nulos e quantidade de textos

In [4]:
df = df[['review_text']].dropna()

df = df.head(500)

print("Quantidade de textos:", len(df))
df.head()

Quantidade de textos: 500


,review_text
0,Bem macio e felpudo...recomendo. Preço imbatí...
1,Produto excepcional! recomendo!!! inovador e ...
2,recebi o produto antes do prazo mas veio com d...
3,Bom custo beneficio. Adequado para pessoas que...
4,Além de higiênico tem o tamanho ideal. Só falt...


Limpeza de letras contendo acento ,e caracteres

### 2 - Limpeza utilizando Regex

In [5]:
def limpar_texto(texto):
  texto = texto.lower()
  texto = re.sub(r'[^a-záàâãéêíóôõúç ]', ' ', texto)
  texto = re.sub(r'\s+',' ', texto)

  return texto.strip()

df['texto_limpo'] = df['review_text'].apply(limpar_texto)
df[['review_text', 'texto_limpo']].head()
df = df.drop(columns = 'review_text')
df.head()

,texto_limpo
0,bem macio e felpudo recomendo preço imbatível ...
1,produto excepcional recomendo inovador e gosto...
2,recebi o produto antes do prazo mas veio com d...
3,bom custo beneficio adequado para pessoas que ...
4,além de higiênico tem o tamanho ideal só falto...


### 3 - Pipeline de pré processamento de texto

A biblioteca utilizada para esse processo foi a biblioteca do spaCy. A biblioteca do SpaCy, após pesquisas realizadas por nós, se mostrou ser a melhor para pré processamento no sentido de aprendizado de uso no mercado pois é uma biblioteca focada na utilização em produção, diferente da biblioteca NLTK que foca puramente no acadêmico.

In [6]:
# Carregando o modelo a ser usado
nlp = spacy.load('pt_core_news_sm')

# Pipeline completo de Pre Processing
def pipeline_completo_spacy(textos):
    resultados = []
    
    # nlp.pipe faz a tokenização inicial automaticamente e em lotes!
    for doc in nlp.pipe(textos, batch_size=50):
        tokens_processados = []

        # Esse loop dentro do loop for já realiza as operações necessárias de pre processamento
        for token in doc:
            # token.is_alpha garante que é texto válido e not token.is_stop rejeita stop words
            if token.is_alpha and not token.is_stop:
                
                # token.lemma_ substitui a sua função aplicar_lemmatizacao
                tokens_processados.append(token.lemma_.lower())
                
        resultados.append(tokens_processados)
        
    return resultados

# Tokens já processados no dataset
df['tokens_lemma'] = pipeline_completo_spacy(df['texto_limpo'])

df[['texto_limpo', 'tokens_lemma']].head()

,texto_limpo,tokens_lemma
0,bem macio e felpudo recomendo preço imbatível ...,"[macio, felpudo, recomendo, preço, imbatível, ..."
1,produto excepcional recomendo inovador e gosto...,"[produto, excepcional, recomendo, inovador, go..."
2,recebi o produto antes do prazo mas veio com d...,"[recebi, produto, prazo, vir, defeito, trar, s..."
3,bom custo beneficio adequado para pessoas que ...,"[custo, beneficio, adequado, pessoa, uso, casu..."
4,além de higiênico tem o tamanho ideal só falto...,"[higiênico, tamanho, ideal, faltar, colher, ga..."


Nota: Os modelos do Spacy como o pt_core_news_sm necessitam de instalação local "python -m spacy download pt_core_news_sm"

### 4 - Motor de Busca Léxico

Fazemos um join nos tokens para eles voltarem a ser strings já que a biblioteca do Skitlearn não trabalha com dados na forma de Tokens.

In [7]:
df['texto_final'] = df['tokens_lemma'].apply(lambda tokens: ' '.join(tokens))
df[['tokens_lemma', 'texto_final']].head()


,tokens_lemma,texto_final
0,"[macio, felpudo, recomendo, preço, imbatível, ...",macio felpudo recomendo preço imbatível entreg...
1,"[produto, excepcional, recomendo, inovador, go...",produto excepcional recomendo inovador gostoso...
2,"[recebi, produto, prazo, vir, defeito, trar, s...",recebi produto prazo vir defeito trar ser amer...
3,"[custo, beneficio, adequado, pessoa, uso, casu...",custo beneficio adequado pessoa uso casual apa...
4,"[higiênico, tamanho, ideal, faltar, colher, ga...",higiênico tamanho ideal faltar colher garfo so...


#### 4.1 - Vetorização

In [15]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
matriz_tfidf = vectorizer.fit_transform(df['texto_final'])
print("Quantidade de documentos:", matriz_tfidf.shape[0])
print("Quantidade de termos:", matriz_tfidf.shape[1])


def buscar_tfidf(consulta, n_resultados=3):
    consulta_processada = preprocessar_consulta_tfidf(consulta)
    vetor_consulta = vectorizer.transform([consulta_processada])
    scores = cosine_similarity(vetor_consulta, matriz_tfidf).ravel()
    indices = scores.argsort()[::-1][:n_resultados]
    return pd.DataFrame({
        'indice': indices,
        'score': scores[indices],
        'documento': df.iloc[indices]['review_text'].values
    })

Quantidade de documentos: 500
Quantidade de termos: 6588


#### 4.2 - Mecanismo de Busca

Funções com o pipeline de pre processamento, utilização do TF-IDF e cálculo de similaridade de cosseno

In [16]:
def preprocessar_consulta_tfidf(consulta):
    consulta_limpa = limpar_texto(consulta)
    tokens = pipeline_completo_spacy([consulta_limpa])[0]
    return ' '.join(tokens)


def buscar_tfidf(consulta, n_resultados=3):
    consulta_processada = preprocessar_consulta_tfidf(consulta)
    vetor_consulta = vectorizer.transform([consulta_processada])

    scores = cosine_similarity(vetor_consulta, matriz_tfidf).ravel()
    indices = scores.argsort()[::-1][:n_resultados]

    return pd.DataFrame({
        'indice': indices,
        'score': scores[indices],
        'documento': df.iloc[indices]['texto_limpo'].values
    })

consulta = "produto bom"
resultados_tfidf = buscar_tfidf(consulta)
resultados_tfidf

,indice,score,documento
0,217,0.217659,gostei muito do produto recomendo entrega rápi...
1,384,0.217286,produto muito bom pratico e rapido eu recomend...
2,295,0.204739,ainda não recebi o produto não tenho como aval...


### 5 - Motor de Busca Semântico

In [49]:
# Treinamento local em português usando os tokens tratados do corpus.
modelo_w2v = Word2Vec(
    sentences=df['tokens_lemma'].tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    workers=1,
    sg=1,
    epochs=30,
    seed=42
)


def vetor_medio(tokens):
    vetores = [modelo_w2v.wv[token] for token in tokens if token in modelo_w2v.wv]
    if not vetores:
        return np.zeros(modelo_w2v.vector_size)
    return np.mean(vetores, axis=0)


vetores_documentos = np.vstack(df['tokens_lemma'].apply(vetor_medio).to_numpy())


def buscar_semantico(consulta, n_resultados=3):
    tokens_consulta = preprocessar_consulta_tfidf(consulta).split()
    tokens_conhecidos = [token for token in tokens_consulta if token in modelo_w2v.wv]

    colunas_resultado = ['indice', 'score', 'documento']
    if not tokens_conhecidos:
        return pd.DataFrame(columns=colunas_resultado)

    vetor_consulta = vetor_medio(tokens_conhecidos).reshape(1, -1)
    scores_semanticos = cosine_similarity(vetor_consulta, vetores_documentos).ravel()

    termos_consulta = set(tokens_consulta)
    cobertura = np.array([
        len(termos_consulta.intersection(tokens_documento)) / len(termos_consulta)
        for tokens_documento in df['tokens_lemma']
    ])

    scores = scores_semanticos * cobertura
    indices = scores.argsort()[::-1][:n_resultados]
    return pd.DataFrame({
        'indice': indices,
        'score': scores[indices],
        'documento': df.iloc[indices]['texto_limpo'].values
    })

print("Dimensão dos vetores densos:", modelo_w2v.vector_size)
print("\nTop 3 resultados semânticos:")
resultados_semanticos = buscar_semantico(consulta)
resultados_semanticos

Dimensão dos vetores densos: 100

Top 3 resultados semânticos:


,indice,score,documento
0,484,0.246955,comprei por aqui pois estava mais barato uso p...
1,230,0.246831,este produto como muitos dizem é uma progressi...
2,422,0.245138,produto não é da mq professional é uma imitaçã...


### TESTES DE CONSULTA NOS DOIS MOTORES

In [54]:
consulta = "ompre na loja de dinossauros"
resultados_tfidf = buscar_tfidf(consulta)

with pd.option_context('display.max_colwidth', None):
    display(resultados_tfidf)

,indice,score,documento
0,372,0.347637,secador muito bom e entrega super rápida lojas americanas a melhor loja pela internet
1,293,0.222539,o produto adquirido corresponde ao ofertado pela loja
2,142,0.215292,produto veio quebrado produto veio imundo tive a impressão que era usado lojatudo nunca mais não tem respeito e muito menos consideração pelos clientes na primeira compra feita enviam o produto quebrado e totalmente sujo loja tudo nota não comprem na loja tudo não compre na loja tudo loja tudo não loja tudo não é boa loja tudo entrega produtos quebrados loja tudo não é confiável


In [55]:
consulta = "Compre na loja de dinossauros"
resultados_semanticos = buscar_semantico(consulta)

with pd.option_context('display.max_colwidth', None):
    display(resultados_semanticos)

,indice,score,documento
0,142,0.646332,produto veio quebrado produto veio imundo tive a impressão que era usado lojatudo nunca mais não tem respeito e muito menos consideração pelos clientes na primeira compra feita enviam o produto quebrado e totalmente sujo loja tudo nota não comprem na loja tudo não compre na loja tudo loja tudo não loja tudo não é boa loja tudo entrega produtos quebrados loja tudo não é confiável
1,293,0.324945,o produto adquirido corresponde ao ofertado pela loja
2,355,0.320711,somos de mg a entrega foi bem tranquila liguei j na loja agradecendo obrigada
